# Landing → Bronze (Auto Loader)

Ingestão das 8 tabelas com **Auto Loader** (`cloudFiles`) em vez de `spark.read.csv` em lote. Continua
sendo um notebook único — a lógica é a mesma pra todas as tabelas, muda só por PR revisado (mesma regra
do notebook base em `analises/`).

⚠️ **`cloudFiles` é específico do Databricks** — não existe no PySpark open-source, então isto **não foi
testado localmente** (diferente de todo o resto do repositório). A primeira execução de verdade precisa
ser no Databricks; reconfira as contagens da seção 4 contra `docs/achados_analise_geral.md` antes de
confiar no resultado.

## Por que Auto Loader aqui, mesmo com arquivo estático (não é um stream de verdade)

- Usamos `.trigger(availableNow=True)`: processa os arquivos que já estão no Volume **uma vez** e para
  — se comporta como um batch, mas com o controle de arquivo-a-arquivo do Auto Loader.
- **Idempotência muda de mecanismo**: não é `overwrite` (como no notebook em lote), é o **checkpoint**
  — Auto Loader só processa cada arquivo uma vez; rodar de novo sem mudar nada não duplica linha.
  Com `cloudFiles.allowOverwrites = true`, se alguém subir de novo um CSV com o mesmo nome (comum
  enquanto estamos testando), o Auto Loader detecta e reprocessa esse arquivo.
- Pra forçar uma reingestão completa de uma tabela (ex.: mudou a estrutura do CSV), apague o checkpoint
  e o schema dela (seção 5) e a tabela de destino, e rode de novo.

Regras da camada bronze que continuam valendo (ver `CLAUDE.md`): tudo como `string` (nenhuma correção de
conteúdo aqui), `_ingestion_ts` e `_source_file` em toda tabela, nenhum `display()`/`show()` de linha
inteira em tabelas com dado pessoal (`clientes`).

## 1. Configuração

In [ ]:
dbutils.widgets.text("base_path", "")
BASE_PATH = dbutils.widgets.get("base_path")
if not BASE_PATH:
    raise ValueError(
        "Preencha o widget 'base_path' com um caminho de Volume: '/Volumes/<catalogo>/<schema>/<volume>'."
    )
if not BASE_PATH.startswith("/Volumes"):
    raise ValueError(
        f"'{BASE_PATH}' não é um Volume. Suba os CSVs num Volume do Unity Catalog e use "
        "'/Volumes/<catalogo>/<schema>/<volume>'."
    )

CATALOGO = "workspace"
SCHEMA_BRONZE = "vcredit_bronze"

# checkpoint e schema do Auto Loader ficam numa subpasta do próprio Volume, uma por tabela
AUTOLOADER_BASE = f"{BASE_PATH}/_autoloader"

## 2. Fontes
Arquivo → (tabela de destino, encoding). `clientes.csv` e `Chamados_Hora.CSV` são `ISO-8859-1`; os
demais `UTF-8`. `Chamados_Hora.CSV` é a única fonte sem aspas.

In [ ]:
SOURCES = {
    "chamados.csv":            ("chamados",            "UTF-8"),
    "Chamados_Hora.CSV":       ("chamados_hora",       "ISO-8859-1"),
    "clientes.csv":            ("clientes",            "ISO-8859-1"),
    "custos.csv":              ("custos",              "UTF-8"),
    "pesquisa_satisfacao.csv": ("pesquisa_satisfacao", "UTF-8"),
    "base_atendentes.csv":     ("base_atendentes",     "UTF-8"),
    "base_motivos.csv":        ("base_motivos",        "UTF-8"),
    "canais.csv":              ("canais",              "UTF-8"),
}

## 3. Ingestão com Auto Loader

Um `readStream`/`writeStream` por tabela, cada um com o seu `checkpointLocation` e `schemaLocation`
(senão o Auto Loader mistura o progresso de tabelas diferentes). `pathGlobFilter` garante que cada
tabela só lê o seu próprio arquivo, mesmo com os 8 CSVs na mesma pasta do Volume.
`cloudFiles.inferColumnTypes = false` mantém tudo `string`, igual ao `inferSchema=False` do notebook em
lote — nenhuma correção de tipo/conteúdo aqui, isso é papel da silver.

In [ ]:
from pyspark.sql import functions as F

resumo = []

for arquivo, (tabela, encoding) in SOURCES.items():
    checkpoint_path = f"{AUTOLOADER_BASE}/{tabela}/checkpoint"
    schema_path = f"{AUTOLOADER_BASE}/{tabela}/schema"

    leitor = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "false")
        .option("cloudFiles.allowOverwrites", "true")
        .option("pathGlobFilter", arquivo)
        .option("header", "true")
        .option("encoding", encoding)
    )
    if arquivo != "Chamados_Hora.CSV":
        leitor = leitor.option("quote", '"').option("escape", '"')

    df = leitor.load(BASE_PATH)

    # padroniza nomes de coluna (remove BOM/espaços à toa, minúsculo) — não muda o conteúdo das células
    for antigo in df.columns:
        novo = antigo.replace("\ufeff", "").strip().lower()
        if novo != antigo:
            df = df.withColumnRenamed(antigo, novo)

    df_bronze = (
        df
        .withColumn("_ingestion_ts", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{tabela}"
    (
        df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(destino)
        .awaitTermination()
    )

    linhas = spark.table(destino).count()
    resumo.append((tabela, arquivo, encoding, linhas))
    print(f"{destino}: {linhas:,} linhas")

## 4. Verificação

Contagens esperadas (arquivos originais, ver `docs/achados_analise_geral.md`): chamados 134.474 ·
chamados_hora 131.838 · clientes 47.705 · custos 131.069 · pesquisa_satisfacao 81.021 ·
base_atendentes 25 · base_motivos 14 · canais 4. Essa célula só mostra contagens — nunca linhas
inteiras (⚠️ `clientes` tem CPF/nome/e-mail).

In [ ]:
display(spark.createDataFrame(resumo, ["tabela", "arquivo", "encoding", "linhas"]))

## 5. Se precisar reprocessar uma tabela do zero

Apague o checkpoint, o schema e a tabela de destino antes de rodar de novo — senão o Auto Loader acha
que já processou tudo e não insere nada:

```python
tabela_para_resetar = "<nome_tabela>"
dbutils.fs.rm(f"{AUTOLOADER_BASE}/{tabela_para_resetar}", recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.{SCHEMA_BRONZE}.{tabela_para_resetar}")
```

## Checklist antes de comitar / dar push

- [ ] Rodei no Databricks pelo menos uma vez — não foi testado localmente (`cloudFiles` não existe fora
      do Databricks).
- [ ] Contagens da seção 4 batem com o esperado.
- [ ] Rodei duas vezes sem apagar checkpoint e confirmei que a segunda vez não duplicou linha.
- [ ] Nenhuma célula de output mostra linha de `clientes` (CPF/nome/e-mail).
- [ ] Se precisou mudar algo, foi por PR revisado — não direto na `main`.